# Compare the brief’s chunking choices

The experiment compares LlamaIndex sentence, token and embedding-based semantic splitting, structure-based sections, whole recipes, and parent-child retrieval. Parent-child indexes sections and returns the full parent recipe; it is not the same as indexing the full recipe directly. We use the same embedding model, 10 known-recipe questions and top-k for each strategy.

The score **recipe hit@5** asks only whether at least one of the five retrieved chunks belongs to the expected recipe. It is not context precision, context recall or faithfulness: the right recipe may still contain the wrong section. The dataset and labels are synthetic, and the 30 short recipes may be too small to reveal meaningful differences. Semantic splitting can produce only one piece for a short recipe. Timing is local and sensitive to caching and hardware. The script reports measured values only when executed; it does not contact Groq.


In [ ]:
from pathlib import Path
import json, sys, time, re
import numpy as np
import pandas as pd
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter, SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

HERE = Path.cwd().resolve()
if not (HERE / 'README.md').exists() or HERE.name != 'comparisons_v1':
    raise RuntimeError('Start the notebook in the v2/comparisons_v1 folder')
V2 = HERE.parent
sys.path.insert(0, str(V2))
from chef.retrieval import read_recipes, make_documents
recipes = read_recipes()
cases = json.loads((V2 / 'data' / 'evaluation_v2.json').read_text(encoding='utf-8'))[:10]
targets = ['Tomato pasta','Mushroom pasta','Lentil soup','Chickpea salad','Carrot soup','Tofu stir fry','Baked potatoes','Pea soup','Chicken rice bowl','Lentil soup']
embed = HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=str(V2.parent / 'storage' / 'models'))

def recipe_text(r):
    items = ', '.join(f"{x['quantity']} {x['unit']} {x['name']}" for x in r['ingredients'])
    return (f"Recipe: {r['name']}. {r['overview']} Serves {r['servings']}. "
            f"Ingredients: {items}. Instructions: {r['instructions']} "
            f"Dietary tags: {', '.join(r['dietary_tags'])}. {r['dietary_note']}")
documents = [Document(text=recipe_text(r), metadata={'recipe': r['name']}) for r in recipes]
parents = {r['name']: recipe_text(r) for r in recipes}

def sentence_split(text):
    return [s for s in re.split(r'(?<=[.!?])\s+', text) if s]

parsers = {
    'sentence': SentenceSplitter.from_defaults(chunk_size=96, chunk_overlap=16, include_metadata=True, chunking_tokenizer_fn=sentence_split),
    'token': TokenTextSplitter.from_defaults(chunk_size=96, chunk_overlap=16, include_metadata=True),
    'semantic': SemanticSplitterNodeParser.from_defaults(embed_model=embed, breakpoint_percentile_threshold=90, include_metadata=True, sentence_splitter=sentence_split),
}
query_vectors = np.asarray(embed.get_text_embedding_batch([c['question'] for c in cases]), dtype=np.float32)
query_vectors /= np.maximum(np.linalg.norm(query_vectors, axis=1, keepdims=True), 1e-12)
rows, details = [], []
for strategy in ['sentence','token','semantic','sections','parent_child','recipe']:
    start = time.perf_counter()
    if strategy in parsers:
        nodes = parsers[strategy].get_nodes_from_documents(documents, show_progress=False)
    elif strategy == 'recipe':
        nodes = documents
    else:
        nodes = make_documents('sections')
    texts = [n.get_content(metadata_mode='none') for n in nodes]
    names = [n.metadata['recipe'] for n in nodes]
    vectors = np.asarray(embed.get_text_embedding_batch(texts), dtype=np.float32)
    vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
    indexing_seconds = time.perf_counter() - start
    start = time.perf_counter()
    ranks = np.argsort(-(query_vectors @ vectors.T), axis=1)[:, :5]
    query_ms = 1000 * (time.perf_counter() - start) / len(cases)
    hits = []
    for case, target, indices in zip(cases, targets, ranks):
        retrieved = [names[i] for i in indices]
        hit = target in retrieved
        hits.append(hit)
        returned_context = [parents[names[i]] if strategy == 'parent_child' else texts[i] for i in indices]
        details.append({'strategy': strategy, 'id':case['id'], 'expected_recipe':target,
                        'retrieved_recipes':retrieved, 'hit':bool(hit),
                        'returned_characters':sum(len(t) for t in returned_context)})
    rows.append({'strategy':strategy, 'chunks':len(texts), 'recipe_hit_at_5':float(np.mean(hits)),
                 'mean_returned_characters':float(np.mean([d['returned_characters'] for d in details[-len(cases):]])),
                 'build_seconds':round(indexing_seconds,2), 'mean_search_ms':round(query_ms,3)})
summary = pd.DataFrame(rows)
detail = pd.DataFrame(details)
display(summary)


## Inspect the cases, not only the mean

The parent-child rows return the full recipe for every matching child; duplicate parents are counted repeatedly in the displayed context size. That makes token cost visible but is not an optimized production parent-child retriever. Build time includes creating and embedding chunks; semantic chunking also embeds sentences to decide boundaries. The lookup timing excludes query embedding and is too small to support meaningful latency claims. For a proper benchmark, run Ragas and measure end-to-end latency and tokens on the same full evaluation set.


In [ ]:
display(detail[['strategy','id','expected_recipe','retrieved_recipes','hit','returned_characters']])
